# Procesamiento del Lenguaje Natural con spaCy

---

## ¿Por qué?

En el último notebook de la sesión de expresiones regulares se formuló una pregunta sin respuesta:

> *`re.findall()` devuelve una lista plana: sin contexto, sin posición, sin información morfosintáctica. ¿Para qué tipo de análisis es suficiente? ¿Para cuál no lo es?*

El patrón `\b\w+ing\b` aplicado sobre el corpus de neologismos devuelve una lista de cadenas que terminan en *-ing*. No sabe si *scrolling* es un sustantivo o un verbo en ese fragmento concreto. No sabe si *mansplaining* tiene marca coloquial en el DRAE. No sabe que *networking* aparece en un contexto de registro laboral y *streaming* en uno de registro formal consolidado. Lo que devuelve es una forma gráfica, no una unidad lingüística.

El **Procesamiento del Lenguaje Natural** (PLN) opera en el nivel siguiente: no sobre caracteres, sino sobre tokens con atributos lingüísticos. Este notebook responde la pregunta que regex dejó abierta.

## ¿Qué?

spaCy es una biblioteca de PLN para Python que aplica sobre un texto una cadena de análisis lingüísticos — un *pipeline* — y convierte cada palabra en un objeto con atributos: categoría gramatical, lema, dependencia sintáctica, pertenencia a una entidad nombrada. Opera sobre el mismo corpus. Produce resultados cualitativamente distintos.

## ¿Para qué?

La sesión de regex extrajo del corpus de neologismos listas de cadenas: términos entre comillas, palabras terminadas en *-ing*, fechas, URLs. Con PLN se puede ir más lejos: identificar qué tipo de entidad es cada nombre propio sin describir su forma, distinguir usos morfológicos del mismo anglicismo, o extraer las palabras con mayor carga semántica del corpus sin conocerlas de antemano.

## ¿Cómo?

El trabajo se articula en cuatro bloques ejecutables sobre el mismo corpus de neologismos de la sesión anterior. El punto de partida es el límite donde regex se detiene.

---
## 1. Instalación y carga del modelo

In [ ]:
!python -m spacy download es_core_news_sm --quiet
print("Modelo descargado.")

In [ ]:
import re
import spacy
from spacy import displacy
from spacy.matcher import Matcher
import pandas as pd
from collections import Counter

modelo_español = spacy.load("es_core_news_sm")
print("Pipeline:", modelo_español.pipe_names)

---
## 2. El mismo corpus

El mismo texto de los notebooks de regex. La diferencia no está en el material sino en la herramienta que lo procesa.

In [ ]:
corpus_neologismos = """
El término "influencer" lleva tres años en los corpus académicos y sigue sin traducción estable.
De acuerdo: "ghosting" ya aparece en el DRAE con marca coloquial.
"woke" pasó del activismo anglosajón al debate público español en menos de dos años. Sin traducción consensuada.
El fenómeno "phishing" no es solo tecnológico: las mismas técnicas de suplantación aparecen en campañas de desinformación lingüística.
"scrollear" ya no necesita explicación. Antes era jerga de redes, ahora aparece en medios generalistas sin comillas.
El problema con "deepfake" es la traducción: ¿ultrafalsificación? ¿simulacro profundo? Ninguna cuaja. El anglicismo gana por defecto.
Términos digitales consolidados en el español formal durante 2024: "streaming", "podcast", "newsletter". Ya en medios de referencia sin comillas ni cursiva.
Sobre "mansplaining": ¿préstamo léxico o calco conceptual? La Academia aún no se ha pronunciado. La presencia en corpus crece año a año.
"Selfie" entró al DRAE en 2014, diez años antes que "ghosting". La velocidad de canonicalización ha aumentado notablemente en la última década.
La palabra "meme" ya aparece en todos los registros sin marca de préstamo.
El "burnout" aparece en informes de recursos humanos, en diagnósticos clínicos y en conversaciones de WhatsApp. Tres registros, un único anglicismo. Sin equivalente estable en español.
¿Es "spoiler" un préstamo o ya es español? La RAE lo recoge como coloquial. El verbo derivado "espoilear" no aparece aún en ningún diccionario normativo.
Búsqueda en CREA: "hater" aparece 43 veces entre 2015 y 2023. "Odiador", propuesta de la Fundéu, aparece 3 veces. Los corpus no mienten.
El debate sobre "fake news" se complica: ¿es una unidad léxica o dos palabras? La Fundéu propone "bulo", con tradición y precisión.
"Trolling" plantea un problema morfológico: el verbo en español sería "trolear" o "troliar". El sustantivo "trol" ya está en el DRAE.
"networking" aparece más en ofertas de empleo en español que en inglés para puestos equivalentes. El anglicismo ha desplazado a "contactos profesionales" en ese registro.
El "spam" pasó de nombre a verbo y a adjetivo en menos de una década. La morfología española absorbió el anglicismo sin resistencia.
"Coaching" y "mentoring" conviven en el mismo documento sin que nadie los distinga. En corpus corporativos de 2023, aparecen juntos en el 67% de los casos.
El "rebranding" de palabras existe: "propaganda" se convirtió en "comunicación estratégica" y "despido" en "desvinculación". El eufemismo también tiene corpus.
La Fundéu recomienda "teletrabajo" sobre "remote work" desde 2020. Cuatro años después, ambos coexisten en corpus periodísticos.
"Hashtag" tiene traducción oficial en francés y propuesta en español. En corpus reales, "hashtag" supera a "etiqueta" en proporción 9:1.
"Storytelling" se ha gramaticalizado como modificador nominal: estrategia de storytelling, taller de storytelling. Aparece en corpus universitarios sin cursiva ni comillas.
Anglicismos del ámbito deportivo con mayor penetración en el español: "ranking", "training", "pressing", "dribbling". Todos con morfología adaptada.
El "crowdfunding" y el "crowdsourcing" comparten prefijo pero no frecuencia: el primero aparece 12 veces más en corpus periodístico.
"""

print(f"Corpus cargado: {len(corpus_neologismos)} caracteres.")

---
## 3. El límite de regex

Punto de partida: lo que regex hace sobre este corpus y lo que no puede hacer.

In [ ]:
# El mismo patrón del notebook colab-01: palabras terminadas en -ing
patron_ing = r"\b\w+ing\b"
resultados_regex = re.findall(patron_ing, corpus_neologismos)

print("Resultado de regex:")
print(resultados_regex)
print(f"\nTotal: {len(resultados_regex)} coincidencias")

El patrón devuelve una lista plana. Todos los resultados son cadenas de caracteres idénticas en estructura: terminan en *-ing*. El patrón no distingue:

- si la palabra es sustantivo, verbo, adjetivo o modificador nominal
- si es anglicismo consolidado o préstamo reciente
- si aparece entre comillas como término citado o en texto corrido como palabra integrada

Para responder esas preguntas hace falta información morfosintáctica, no gráfica.

---
## 4. El mismo corpus procesado con spaCy

spaCy convierte el texto en un `Doc`: una secuencia de tokens, cada uno con una ficha de atributos lingüísticos calculados automáticamente.

In [ ]:
documento_corpus = modelo_español(corpus_neologismos)
print(f"Tokens en el corpus: {len(documento_corpus)}")

In [ ]:
# Los mismos términos -ing, ahora con atributos lingüísticos
ETIQUETAS_ESPAÑOL = {
    "ADJ":   "Adjetivo",
    "ADP":   "Preposición",
    "ADV":   "Adverbio",
    "AUX":   "Verbo auxiliar",
    "CCONJ": "Conjunción coordinante",
    "DET":   "Determinante",
    "NOUN":  "Sustantivo",
    "PROPN": "Nombre propio",
    "VERB":  "Verbo",
    "X":     "Otro / no clasificado",
}

terminos_ing = [
    {
        "Forma": token.text,
        "Lema": token.lemma_,
        "Categoría": ETIQUETAS_ESPAÑOL.get(token.pos_, token.pos_),
        "Oración": token.sent.text.strip()[:80] + "...",
    }
    for token in documento_corpus
    if token.text.lower().endswith("ing") and len(token.text) > 3
]

pd.DataFrame(terminos_ing)

La misma lista de formas gráficas que regex, pero ahora cada una tiene categoría gramatical y contexto de oración. Es posible distinguir los que aparecen como sustantivos, los que funcionan como modificadores nominales (*taller de storytelling*) y los que el modelo no clasifica con precisión — columna *Otro / no clasificado* — por ser préstamos sin equivalente morfológico en español.

---
## 5. El Matcher: patrones sobre tokens, no sobre caracteres

spaCy incluye un componente llamado `Matcher` que funciona con la misma lógica que regex — definir un patrón, buscar coincidencias — pero opera sobre atributos de tokens en lugar de sobre caracteres. En lugar de describir cómo se escribe una palabra, se describe qué función cumple.

In [ ]:
# Comparación directa
# ─────────────────────────────────────────────────────────────────
# Regex:   r"\b\w+ing\b"
#          → cualquier cadena que termina en -ing
#
# Matcher: [{"TEXT": {"REGEX": "\\w+ing"}}, {"POS": "NOUN"}]
#          → token que termina en -ing seguido de un sustantivo
#            (estructura «X de storytelling», «taller de coaching»)
# ─────────────────────────────────────────────────────────────────

buscador = Matcher(modelo_español.vocab)

# Patrón 1: anglicismo -ing usado como sustantivo directamente
patron_anglicismo_sustantivo = [{"TEXT": {"REGEX": "\\w+ing"}, "POS": "NOUN"}]

# Patrón 2: estructura «de + anglicismo -ing» (modificador nominal)
patron_modificador_nominal = [
    {"LOWER": "de"},
    {"TEXT": {"REGEX": "\\w+ing"}},
]

buscador.add("ANGLICISMO_SUSTANTIVO", [patron_anglicismo_sustantivo])
buscador.add("MODIFICADOR_NOMINAL",   [patron_modificador_nominal])

coincidencias = buscador(documento_corpus)

filas_matcher = []
for identificador_patron, inicio, fin in coincidencias:
    nombre_patron = modelo_español.vocab.strings[identificador_patron]
    fragmento = documento_corpus[inicio:fin]
    filas_matcher.append({
        "Patrón": nombre_patron,
        "Fragmento encontrado": fragmento.text,
        "Contexto": fragmento.sent.text.strip()[:80] + "...",
    })

pd.DataFrame(filas_matcher)

El Matcher no amplía lo que regex hace: lo eleva a otro nivel de descripción. En lugar de preguntar *¿termina esta cadena en -ing?*, pregunta *¿funciona este token como sustantivo? ¿aparece en una construcción de modificador nominal?*. La diferencia es la misma que hay entre describir la forma gráfica de una palabra y describir su función en la oración.

---
## 6. Reconocimiento de entidades nombradas

El NER identifica y clasifica automáticamente los nombres propios del texto. No hace falta describir su forma gráfica: el modelo los reconoce por su función y contexto.

In [ ]:
# En regex, localizar «DRAE», «Fundéu», «RAE», «CREA» requería un patrón
# de siglas: r'\b[A-Z]{2,}\b' — y capturaba también ruido.
# El NER los identifica directamente como organizaciones.

TIPOS_ENTIDAD_ESPAÑOL = {
    "PER":  "Persona",
    "LOC":  "Lugar",
    "ORG":  "Organización",
    "MISC": "Otro nombre propio",
}

lista_entidades = [
    {
        "Entidad": entidad.text,
        "Tipo": TIPOS_ENTIDAD_ESPAÑOL.get(entidad.label_, entidad.label_),
        "Contexto": entidad.sent.text.strip()[:80] + "...",
    }
    for entidad in documento_corpus.ents
]

tabla_entidades = pd.DataFrame(lista_entidades)
tabla_entidades

In [ ]:
# Visualización con colores sobre una oración concreta
oracion_instituciones = modelo_español(
    "Búsqueda en CREA: 'hater' aparece 43 veces entre 2015 y 2023. "
    "'Odiador', propuesta de la Fundéu, aparece 3 veces. "
    "La RAE lo recoge como coloquial. El verbo derivado no aparece en ningún diccionario normativo."
)
displacy.render(oracion_instituciones, style="ent", jupyter=True)

---
## 7. Análisis de frecuencias con información morfológica

En regex, la frecuencia de términos se obtenía con `Counter` sobre la lista plana devuelta por `re.findall()`. Los términos eran cadenas: no había forma de agrupar variantes morfológicas del mismo elemento.

Con spaCy, se trabaja sobre lemas. *anglicismo*, *anglicismos* y *anglicismo* son el mismo lema: se cuentan juntos.

In [ ]:
import matplotlib.pyplot as plt

# Solo sustantivos y nombres propios, sin stopwords ni puntuación
lemas_con_carga_semantica = [
    token.lemma_.lower()
    for token in documento_corpus
    if token.pos_ in ("NOUN", "PROPN")
    and not token.is_stop
    and not token.is_punct
    and len(token.text) > 2
]

conteo_lemas = Counter(lemas_con_carga_semantica)
top_20 = conteo_lemas.most_common(20)

palabras_top, frecuencias_top = zip(*top_20)

fig, eje = plt.subplots(figsize=(10, 6))
colores_barras = ["#2e86ab" if frecuencia > 5 else "#a8dadc" for frecuencia in frecuencias_top]
barras = eje.barh(list(palabras_top)[::-1], list(frecuencias_top)[::-1], color=colores_barras[::-1])

eje.set_xlabel("Número de apariciones (por lema)", fontsize=11)
eje.set_title("Términos con mayor carga semántica\nCorpus de neologismos del español digital",
              fontsize=13, fontweight="bold")
eje.spines["top"].set_visible(False)
eje.spines["right"].set_visible(False)

for barra, frecuencia in zip(barras, list(frecuencias_top)[::-1]):
    eje.text(barra.get_width() + 0.1, barra.get_y() + barra.get_height() / 2,
             str(frecuencia), va="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Comparación: regex vs. lematización
# ¿Cuántas formas distintas de «anglicismo» hay en el corpus?

formas_regex = re.findall(r"\banglicismo\w*\b", corpus_neologismos, re.IGNORECASE)
formas_spacy = [
    token.text for token in documento_corpus
    if token.lemma_.lower() == "anglicismo"
]

print("Formas encontradas por regex:", sorted(set(formas_regex)))
print("Formas encontradas por spaCy:", sorted(set(formas_spacy)))
print()
print(f"regex cuenta: {len(formas_regex)} ocurrencias")
print(f"spaCy agrupa bajo el lema 'anglicismo': {len(formas_spacy)} ocurrencias")

---
## ¿Y ahora qué?

### La escalera completa

Las tres sesiones recorren una progresión en el nivel de análisis sobre el mismo tipo de material textual:

| Sesión | Herramienta | Pregunta que responde | Unidad de análisis |
|---|---|---|---|
| Regex | `re` / Sheets | ¿Coincide esta forma gráfica con el patrón? | Caracteres |
| PLN (esta sesión) | spaCy `Matcher` | ¿Cumple este token esta función lingüística? | Tokens con atributos |
| PLN (esta sesión) | spaCy NER | ¿Qué tipo de entidad es este grupo de tokens? | Spans tipados |
| Próximas sesiones | Embeddings / transformers | ¿Qué significa esto en este contexto? | Vectores semánticos |

Cada nivel no reemplaza al anterior: regex sigue siendo la herramienta adecuada para validar formatos, limpiar ruido y normalizar texto. spaCy resuelve los problemas que regex estructuralmente no puede: función gramatical, tipo de entidad, agrupación por lema.

### Lo que sigue en PLN

| Técnica | Qué añade | Aplicación en el corpus de neologismos |
|---|---|---|
| **Similitud semántica** | Distancia de significado entre palabras o documentos | Agrupar anglicismos semánticamente próximos aunque tengan formas distintas |
| **Análisis de dependencias** | Estructura sintáctica de cada oración | Extraer tripletas *sujeto–verbo–objeto* («La Fundéu recomienda teletrabajo») |
| **Clasificación de texto** | Asignar categorías a fragmentos completos | Separar fragmentos descriptivos de fragmentos evaluativos en el corpus |
| **Embeddings / transformers** | Representación vectorial del significado en contexto | Detectar que *bulo* y *fake news* son equivalentes semánticos sin regla explícita |

### El ejercicio siguiente

Se trabajará con un corpus más amplio de artículos de prensa en español. El objetivo será construir una red de co-ocurrencias: qué organizaciones, instituciones y términos aparecen mencionados en el mismo fragmento con mayor frecuencia. Ese tipo de red es la base de los sistemas de análisis de influencia institucional y de las herramientas de investigación periodística asistida.